In [3]:
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from category_encoders import TargetEncoder

import torch
import torch.nn as nn

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

DATA_DIR = PROJECT_ROOT / "data"
OOF_DIR = PROJECT_ROOT / "oof_preds"
SUB_DIR = PROJECT_ROOT / "submissions"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

TARGET = "임신 성공 여부"

FOLD_SEED = 42
N_SPLITS = 5
POS_WEIGHT = 190123 / 66228

MODEL_SEED = 2024

SAVE_DIR = OOF_DIR / "combo_te_v1_mlp_seed2024"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUB_SAVE_DIR = SUB_DIR / "mlp_blend"
SUB_SAVE_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

PREVIOUS_MLP_BEST_OOF = 0.7409117138464961
CHAMPION_4SEED_OOF = 0.7407705074934163

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_PATH exists:", TRAIN_PATH.exists())
print("SAVE_DIR:", SAVE_DIR)
print("DEVICE:", DEVICE)

PROJECT_ROOT: /mnt/c/dev/my_ml_project
TRAIN_PATH exists: True
SAVE_DIR: /mnt/c/dev/my_ml_project/oof_preds/combo_te_v1_mlp_seed2024
DEVICE: cuda


In [4]:
def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [5]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def rank01(pred):
    return rankdata(pred) / len(pred)

In [6]:
def add_combo_columns(X):
    X = X.copy()

    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = X[col1].astype(str) + "_" + X[col2].astype(str)
            combo_cols.append(new_col)

    return X, combo_cols


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10, random_state=42):
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder

In [7]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int).reset_index(drop=True)
X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

X_raw = X_raw.drop(columns=id_cols, errors="ignore").reset_index(drop=True)
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore").reset_index(drop=True)

X = data_preprocessing(X_raw)
X_test = data_preprocessing(X_test_raw)

X_combo, combo_cols = add_combo_columns(X)
X_test_combo, _ = add_combo_columns(X_test)

base_te_cols = [
    "시술 시기 코드",
    "시술 유형",
    "특정 시술 유형",
    "배란 유도 유형",
    "난자 출처",
    "정자 출처",
    "배아 생성 주요 이유",
    "시술 당시 나이",
]

base_te_cols = [col for col in base_te_cols if col in X_combo.columns]
te_cols = base_te_cols + combo_cols

print("base_te_cols:", base_te_cols)
print("combo_cols:", combo_cols)
print("te_cols count:", len(te_cols))

X_te_combo, te_encoder_combo = add_oof_target_encoding(
    X_combo,
    y,
    cols=te_cols,
    n_splits=N_SPLITS,
    smoothing=10,
    random_state=FOLD_SEED
)

test_te_values = te_encoder_combo.transform(X_test_combo[te_cols])

for col in te_cols:
    X_test_combo[f"{col}_TE"] = test_te_values[col].values

X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")
X_test_te_combo = X_test_combo.drop(columns=combo_cols, errors="ignore")

X_test_te_combo = X_test_te_combo[X_te_combo.columns]

X_stack = X_te_combo.copy().reset_index(drop=True)
X_test_stack = X_test_te_combo.copy().reset_index(drop=True)
y_stack = y.reset_index(drop=True)

print("X_stack:", X_stack.shape)
print("X_test_stack:", X_test_stack.shape)
print("y_stack:", y_stack.shape)

assert list(X_stack.columns) == list(X_test_stack.columns)
assert len(X_stack) == len(y_stack)
assert len(X_test_stack) == len(submission)

base_te_cols: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이']
combo_cols: ['시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
te_cols count: 18
Target Encoding Fold 1
Target Encoding Fold 2
Target Encoding Fold 3
Target Encoding Fold 4
Target Encoding Fold 5
X_stack: (256351, 110)
X_test_stack: (90067, 110)
y_stack: (256351,)


In [8]:
def make_mlp_preprocessor(X):
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    print("numeric_features:", len(numeric_features))
    print("categorical_features:", len(categorical_features))

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=20,
                dtype=np.float32,
            )
        ),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )

    return preprocessor

In [9]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(512, 128), dropout=0.25):
        super().__init__()

        layers = []
        prev_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim

        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


def sparse_to_dense_tensor(X_sparse_batch):
    if hasattr(X_sparse_batch, "toarray"):
        X_dense = X_sparse_batch.toarray().astype(np.float32)
    else:
        X_dense = np.asarray(X_sparse_batch, dtype=np.float32)

    return torch.from_numpy(X_dense)


def predict_mlp(model, X_mat, batch_size=8192, device="cpu"):
    model.eval()
    preds = []

    n = X_mat.shape[0]

    with torch.no_grad():
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            xb = sparse_to_dense_tensor(X_mat[start:end]).to(device)
            logits = model(xb)
            prob = torch.sigmoid(logits).detach().cpu().numpy()
            preds.append(prob)

    return np.concatenate(preds)

In [10]:
def train_one_fold_mlp(
    X_tr_mat,
    y_tr,
    X_val_mat,
    y_val,
    input_dim,
    seed=42,
    max_epochs=30,
    batch_size=4096,
    lr=1e-3,
    weight_decay=1e-4,
    patience=5,
    device="cpu",
):
    set_seed(seed)

    model = TabularMLP(
        input_dim=input_dim,
        hidden_dims=(512, 128),
        dropout=0.25
    ).to(device)

    pos_weight_tensor = torch.tensor([POS_WEIGHT], dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    y_tr_np = np.asarray(y_tr, dtype=np.float32)
    y_val_np = np.asarray(y_val, dtype=np.float32)

    best_auc = -np.inf
    best_state = None
    best_epoch = 0
    no_improve = 0

    n_train = X_tr_mat.shape[0]

    for epoch in range(1, max_epochs + 1):
        model.train()

        perm = np.random.permutation(n_train)
        epoch_losses = []

        for start in range(0, n_train, batch_size):
            batch_idx = perm[start:start + batch_size]

            xb = sparse_to_dense_tensor(X_tr_mat[batch_idx]).to(device)
            yb = torch.from_numpy(y_tr_np[batch_idx]).to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

        val_pred = predict_mlp(
            model,
            X_val_mat,
            batch_size=batch_size,
            device=device
        )

        val_auc = roc_auc_score(y_val_np, val_pred)
        scheduler.step(val_auc)

        mean_loss = float(np.mean(epoch_losses))

        print(
            f"Epoch {epoch:02d} | "
            f"loss {mean_loss:.5f} | "
            f"val_auc {val_auc:.6f}"
        )

        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)

    return model, best_auc, best_epoch

In [11]:
def train_mlp_oof_test(
    X_stack,
    X_test_stack,
    y_stack,
    n_splits=5,
    fold_seed=42,
    model_seed=2024,
    save_dir=None,
):
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=fold_seed
    )

    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    fold_scores = []
    best_epochs = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print("\n" + "=" * 100)
        print(f"MLP model_seed={model_seed} Fold {fold}")
        print("=" * 100)

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx].values
        y_val = y_stack.iloc[val_idx].values

        preprocessor = make_mlp_preprocessor(X_tr)

        X_tr_mat = preprocessor.fit_transform(X_tr).astype(np.float32)
        X_val_mat = preprocessor.transform(X_val).astype(np.float32)
        X_test_mat = preprocessor.transform(X_test_stack).astype(np.float32)

        input_dim = X_tr_mat.shape[1]

        print("X_tr_mat:", X_tr_mat.shape)
        print("X_val_mat:", X_val_mat.shape)
        print("X_test_mat:", X_test_mat.shape)
        print("input_dim:", input_dim)

        model, best_auc, best_epoch = train_one_fold_mlp(
            X_tr_mat=X_tr_mat,
            y_tr=y_tr,
            X_val_mat=X_val_mat,
            y_val=y_val,
            input_dim=input_dim,
            seed=model_seed + fold,
            max_epochs=30,
            batch_size=4096,
            lr=1e-3,
            weight_decay=1e-4,
            patience=5,
            device=DEVICE,
        )

        val_pred = predict_mlp(
            model,
            X_val_mat,
            batch_size=8192,
            device=DEVICE
        )

        test_fold_pred = predict_mlp(
            model,
            X_test_mat,
            batch_size=8192,
            device=DEVICE
        )

        fold_auc = roc_auc_score(y_val, val_pred)

        print(f"MLP seed {model_seed} Fold {fold} AUC:", fold_auc)
        print("best_auc during training:", best_auc)
        print("best_epoch:", best_epoch)

        oof[val_idx] = val_pred
        test_pred += test_fold_pred / n_splits

        fold_scores.append(fold_auc)
        best_epochs.append(best_epoch)

        if save_dir is not None:
            np.save(save_dir / f"mlp_seed{model_seed}_partial_oof_fold{fold}.npy", oof)
            np.save(save_dir / f"mlp_seed{model_seed}_partial_test_fold{fold}.npy", test_pred)

    oof_auc = roc_auc_score(y_stack, oof)

    print("\n" + "=" * 100)
    print(f"MLP seed {model_seed} RESULT")
    print("fold_scores:", fold_scores)
    print("mean fold auc:", np.mean(fold_scores))
    print("OOF AUC:", oof_auc)
    print("best_epochs:", best_epochs)
    print("=" * 100)

    if save_dir is not None:
        np.save(save_dir / f"mlp_seed{model_seed}_oof.npy", oof)
        np.save(save_dir / f"mlp_seed{model_seed}_test_pred.npy", test_pred)
        np.save(save_dir / "y_stack.npy", y_stack.to_numpy())

        pd.DataFrame([{
            "model": "mlp",
            "fold_seed": fold_seed,
            "model_seed": model_seed,
            "oof_auc": oof_auc,
            "mean_fold_auc": np.mean(fold_scores),
            "fold_scores": str(fold_scores),
            "best_epochs": str(best_epochs),
        }]).to_csv(
            save_dir / f"mlp_seed{model_seed}_summary.csv",
            index=False
        )

    return oof, test_pred, fold_scores

In [12]:
mlp2024_oof, mlp2024_test_pred, mlp2024_fold_scores = train_mlp_oof_test(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    n_splits=N_SPLITS,
    fold_seed=FOLD_SEED,
    model_seed=MODEL_SEED,
    save_dir=SAVE_DIR,
)

print("MLP seed2024 OOF:", roc_auc_score(y_stack, mlp2024_oof))


MLP model_seed=2024 Fold 1
numeric_features: 56
categorical_features: 54
X_tr_mat: (205080, 256)
X_val_mat: (51271, 256)
X_test_mat: (90067, 256)
input_dim: 256
Epoch 01 | loss 0.87885 | val_auc 0.733340
Epoch 02 | loss 0.85729 | val_auc 0.734979
Epoch 03 | loss 0.85517 | val_auc 0.735010
Epoch 04 | loss 0.85355 | val_auc 0.735701
Epoch 05 | loss 0.85306 | val_auc 0.736033
Epoch 06 | loss 0.85083 | val_auc 0.735803
Epoch 07 | loss 0.84984 | val_auc 0.735572
Epoch 08 | loss 0.85023 | val_auc 0.734917
Epoch 09 | loss 0.84702 | val_auc 0.735844
Epoch 10 | loss 0.84710 | val_auc 0.735889
Early stopping at epoch 10
MLP seed 2024 Fold 1 AUC: 0.7360332644303422
best_auc during training: 0.736033268401124
best_epoch: 5

MLP model_seed=2024 Fold 2
numeric_features: 56
categorical_features: 54
X_tr_mat: (205081, 257)
X_val_mat: (51270, 257)
X_test_mat: (90067, 257)
input_dim: 257
Epoch 01 | loss 0.87997 | val_auc 0.738274
Epoch 02 | loss 0.85940 | val_auc 0.739363
Epoch 03 | loss 0.85825 | val_

In [13]:
# 기존 MLP seed42
MLP42_DIR = OOF_DIR / "combo_te_v1_mlp_seed42"

mlp42_oof = np.load(MLP42_DIR / "mlp_oof.npy")
mlp42_test_pred = np.load(MLP42_DIR / "mlp_test_pred.npy")

print("MLP seed42 OOF:", roc_auc_score(y_stack, mlp42_oof))
print("MLP seed2024 OOF:", roc_auc_score(y_stack, mlp2024_oof))

MLP seed42 OOF: 0.7381970877751112
MLP seed2024 OOF: 0.7382217325622167


In [14]:
# Champion 4-seed OOF
seed42_oof = np.load(OOF_DIR / "combo_te_s10" / "final_combo_rank.npy")
seed2024_oof = np.load(OOF_DIR / "combo_te_v1_s10_seed2024" / "final_oof_seed2024.npy")
seed777_oof = np.load(OOF_DIR / "combo_te_v1_s10_seed777" / "final_oof_seed777.npy")
seed999_oof = np.load(OOF_DIR / "combo_te_v1_s10_seed999" / "final_oof_seed999.npy")

champion_4seed_oof = (
    seed42_oof +
    seed2024_oof +
    seed777_oof +
    seed999_oof
) / 4

champion_4seed_score = roc_auc_score(y_stack, champion_4seed_oof)

print("Champion 4-seed OOF:", champion_4seed_score)

Champion 4-seed OOF: 0.7407705074934163


In [15]:
# Champion 4-seed test
champion_4seed_test = np.load(
    SUB_DIR
    / "seed_ensemble"
    / "final_pred_combo_te_v1_seed42_2024_777_999_avg.npy"
)

print("champion_4seed_test:", champion_4seed_test.shape)

champion_4seed_test: (90067,)


In [16]:
mlp_avg_oof = (
    mlp42_oof +
    mlp2024_oof
) / 2

mlp_avg_test_pred = (
    mlp42_test_pred +
    mlp2024_test_pred
) / 2

print("MLP seed42 OOF:", roc_auc_score(y_stack, mlp42_oof))
print("MLP seed2024 OOF:", roc_auc_score(y_stack, mlp2024_oof))
print("MLP avg 42+2024 OOF:", roc_auc_score(y_stack, mlp_avg_oof))

np.save(SAVE_DIR / "mlp_avg_seed42_2024_oof.npy", mlp_avg_oof)
np.save(SAVE_DIR / "mlp_avg_seed42_2024_test_pred.npy", mlp_avg_test_pred)

MLP seed42 OOF: 0.7381970877751112
MLP seed2024 OOF: 0.7382217325622167
MLP avg 42+2024 OOF: 0.7387467936612893


In [17]:
def search_champion_mlp_blend(
    y_true,
    champion_oof,
    mlp_oof,
    champion_test,
    mlp_test,
    candidate_name,
    out_dir,
):
    champ_rank_oof = rank01(champion_oof)
    mlp_rank_oof = rank01(mlp_oof)

    champion_score = roc_auc_score(y_true, champion_oof)
    mlp_score = roc_auc_score(y_true, mlp_oof)

    best_score = champion_score
    best_w = 0.0
    best_blend_oof = champion_oof.copy()

    rows = []

    for w_mlp in np.arange(0.00, 0.31, 0.01):
        blend_oof = (
            (1 - w_mlp) * champ_rank_oof +
            w_mlp * mlp_rank_oof
        )

        score = roc_auc_score(y_true, blend_oof)

        rows.append({
            "candidate": candidate_name,
            "w_mlp": w_mlp,
            "blend_oof": score,
            "champion_oof": champion_score,
            "mlp_oof": mlp_score,
            "improvement": score - champion_score,
        })

        if score > best_score:
            best_score = score
            best_w = w_mlp
            best_blend_oof = blend_oof.copy()

    result_df = pd.DataFrame(rows).sort_values("blend_oof", ascending=False)

    np.save(out_dir / f"{candidate_name}_best_blend_oof_w{best_w:.2f}.npy", best_blend_oof)
    result_df.to_csv(out_dir / f"{candidate_name}_blend_search.csv", index=False)

    print("\n" + "=" * 80)
    print("candidate:", candidate_name)
    print("champion_oof:", champion_score)
    print("mlp_oof:", mlp_score)
    print("best_blend_oof:", best_score)
    print("best_w_mlp:", best_w)
    print("improvement:", best_score - champion_score)
    print("=" * 80)

    return {
        "candidate": candidate_name,
        "champion_oof": champion_score,
        "mlp_oof": mlp_score,
        "best_blend_oof": best_score,
        "best_w_mlp": best_w,
        "improvement": best_score - champion_score,
        "best_blend_oof_array": best_blend_oof,
        "blend_search_df": result_df,
        "mlp_test": mlp_test,
    }

In [18]:
blend_results = []

res_mlp42 = search_champion_mlp_blend(
    y_true=y_stack,
    champion_oof=champion_4seed_oof,
    mlp_oof=mlp42_oof,
    champion_test=champion_4seed_test,
    mlp_test=mlp42_test_pred,
    candidate_name="mlp_seed42",
    out_dir=SAVE_DIR,
)

blend_results.append(res_mlp42)

res_mlp2024 = search_champion_mlp_blend(
    y_true=y_stack,
    champion_oof=champion_4seed_oof,
    mlp_oof=mlp2024_oof,
    champion_test=champion_4seed_test,
    mlp_test=mlp2024_test_pred,
    candidate_name="mlp_seed2024",
    out_dir=SAVE_DIR,
)

blend_results.append(res_mlp2024)

res_mlp_avg = search_champion_mlp_blend(
    y_true=y_stack,
    champion_oof=champion_4seed_oof,
    mlp_oof=mlp_avg_oof,
    champion_test=champion_4seed_test,
    mlp_test=mlp_avg_test_pred,
    candidate_name="mlp_avg_seed42_2024",
    out_dir=SAVE_DIR,
)

blend_results.append(res_mlp_avg)


candidate: mlp_seed42
champion_oof: 0.7407705074934163
mlp_oof: 0.7381970877751112
best_blend_oof: 0.7409117138464961
best_w_mlp: 0.18
improvement: 0.00014120635307979246

candidate: mlp_seed2024
champion_oof: 0.7407705074934163
mlp_oof: 0.7382217325622167
best_blend_oof: 0.7408754335993506
best_w_mlp: 0.17
improvement: 0.00010492610593426654

candidate: mlp_avg_seed42_2024
champion_oof: 0.7407705074934163
mlp_oof: 0.7387467936612893
best_blend_oof: 0.74091342083597
best_w_mlp: 0.21
improvement: 0.00014291334255367438


In [20]:
blend_summary = pd.DataFrame([
    {
        "candidate": r["candidate"],
        "champion_oof": r["champion_oof"],
        "mlp_oof": r["mlp_oof"],
        "best_blend_oof": r["best_blend_oof"],
        "best_w_mlp": r["best_w_mlp"],
        "improvement": r["improvement"],
        "beats_previous_mlp_best": r["best_blend_oof"] - PREVIOUS_MLP_BEST_OOF,
    }
    for r in blend_results
]).sort_values("best_blend_oof", ascending=False)

display(blend_summary)

blend_summary.to_csv(SAVE_DIR / "mlp_seed_blend_summary.csv", index=False)

,candidate,champion_oof,mlp_oof,best_blend_oof,best_w_mlp,improvement,beats_previous_mlp_best
2,mlp_avg_seed42_2024,0.740771,0.738747,0.740913,0.21,0.000143,0.000002
0,mlp_seed42,0.740771,0.738197,0.740912,0.18,0.000141,0.000000
1,mlp_seed2024,0.740771,0.738222,0.740875,0.17,0.000105,-0.000036


In [21]:
best_row = blend_summary.iloc[0]
best_candidate = best_row["candidate"]
best_w = float(best_row["best_w_mlp"])
best_oof = float(best_row["best_blend_oof"])
best_improvement = float(best_row["improvement"])

print("best_candidate:", best_candidate)
print("best_w:", best_w)
print("best_oof:", best_oof)
print("best_improvement:", best_improvement)
print("previous_mlp_best:", PREVIOUS_MLP_BEST_OOF)
print("beats previous:", best_oof - PREVIOUS_MLP_BEST_OOF)

result_map = {r["candidate"]: r for r in blend_results}
best_result = result_map[best_candidate]
best_mlp_test = best_result["mlp_test"]

if best_w > 0:
    champion_rank_test = rank01(champion_4seed_test)
    mlp_rank_test = rank01(best_mlp_test)

    final_pred = (
        (1 - best_w) * champion_rank_test +
        best_w * mlp_rank_test
    )

    submission_out = pd.read_csv(SUBMISSION_PATH)
    pred_col = submission_out.columns[-1]
    submission_out[pred_col] = final_pred

    out_csv = SUB_SAVE_DIR / f"submission_champion4seed_{best_candidate}_w{best_w:.2f}.csv"
    out_npy = SUB_SAVE_DIR / f"final_pred_champion4seed_{best_candidate}_w{best_w:.2f}.npy"

    submission_out.to_csv(out_csv, index=False)
    np.save(out_npy, final_pred)

    summary_out = pd.DataFrame([{
        "candidate": best_candidate,
        "champion_4seed_oof": champion_4seed_score,
        "mlp_oof": float(best_row["mlp_oof"]),
        "blend_oof": best_oof,
        "w_champion": 1 - best_w,
        "w_mlp": best_w,
        "improvement": best_improvement,
        "previous_mlp_best_oof": PREVIOUS_MLP_BEST_OOF,
        "beats_previous_mlp_best": best_oof - PREVIOUS_MLP_BEST_OOF,
        "file": out_csv.name,
    }])

    summary_out.to_csv(
        SUB_SAVE_DIR / f"summary_champion4seed_{best_candidate}_w{best_w:.2f}.csv",
        index=False
    )

    print("saved csv:", out_csv)
    print("saved npy:", out_npy)
    print(submission_out[pred_col].describe())

else:
    print("No positive MLP blend. Submission not saved.")

best_candidate: mlp_avg_seed42_2024
best_w: 0.21
best_oof: 0.74091342083597
best_improvement: 0.00014291334255367438
previous_mlp_best: 0.7409117138464961
beats previous: 1.7069894738819258e-06
saved csv: /mnt/c/dev/my_ml_project/submissions/mlp_blend/submission_champion4seed_mlp_avg_seed42_2024_w0.21.csv
saved npy: /mnt/c/dev/my_ml_project/submissions/mlp_blend/final_pred_champion4seed_mlp_avg_seed42_2024_w0.21.npy
count    90067.000000
mean         0.500006
std          0.288039
min          0.000092
25%          0.251360
50%          0.500771
75%          0.748627
max          0.999995
Name: probability, dtype: float64


In [22]:
csv_path = out_csv
npy_path = out_npy
sample_path = SUBMISSION_PATH

sub = pd.read_csv(csv_path)
sample = pd.read_csv(sample_path)
pred = np.load(npy_path)

pred_col = sub.columns[-1]

print("csv exists:", csv_path.exists())
print("npy exists:", npy_path.exists())
print("same columns:", list(sub.columns) == list(sample.columns))
print("same length:", len(sub) == len(sample))
print("finite:", np.isfinite(sub[pred_col]).all())
print("min >= 0:", sub[pred_col].min() >= 0)
print("max <= 1:", sub[pred_col].max() <= 1)
print("npy/csv same:", np.allclose(pred, sub[pred_col].values))
print(sub[pred_col].describe())

csv exists: True
npy exists: True
same columns: True
same length: True
finite: True
min >= 0: True
max <= 1: True
npy/csv same: True
count    90067.000000
mean         0.500006
std          0.288039
min          0.000092
25%          0.251360
50%          0.500771
75%          0.748627
max          0.999995
Name: probability, dtype: float64
